# Chapter 4: A basic ReAct agent
Classes live in `src/scratch_agents/`; this notebook uses and tests them.

## Goal (listing 4.1)
By the end of 4.6, this should run:

```python
from scratch_agents import Agent, LlmClient
from scratch_agents.tools import calculator, search_web

agent = Agent(
    model=LlmClient(model="gpt-5-mini"),
    tools=[calculator, search_web],
    instructions="You are a helpful assistant"
)

result = await agent.run("What is 1234 * 5678?")
```

## Setup
Autoreload re-reads my package files when I edit them, so no kernel restart is needed.

In [1]:
%load_ext autoreload
%autoreload 2

## 4.3 Content types (listing 4.2)
Quick check that Message builds and fills in its `type` discriminator.

In [3]:
from scratch_agents.models import Message, ToolCall, ToolResult

m = Message(role="user", content="What is 1234 * 5678?")
print(m)
print(m.type)
print(m.model_dump())

type='message' role='user' content='What is 1234 * 5678?'
message
{'type': 'message', 'role': 'user', 'content': 'What is 1234 * 5678?'}


### Break it on purpose: Pydantic validation

In [4]:
Message(role="admin", content="hi")          # a. role not in the Literal

ValidationError: 1 validation error for Message
role
  Input should be 'system', 'user' or 'assistant' [type=literal_error, input_value='admin', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error

In [5]:
Message(role="user")                          # b. content missing

ValidationError: 1 validation error for Message
content
  Field required [type=missing, input_value={'role': 'user'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

In [6]:
ToolCall(tool_call_id="c1", name="calculator",
         arguments='{"expression": "1+1"}')  # c. arguments is a string, not a dict

ValidationError: 1 validation error for ToolCall
arguments
  Input should be a valid dictionary [type=dict_type, input_value='{"expression": "1+1"}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/dict_type

In [7]:
Message(type="tool_call", role="user", content="hi")   # d. tries to fake the discriminator

ValidationError: 1 validation error for Message
type
  Input should be 'message' [type=literal_error, input_value='tool_call', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error

## 4.3 Event and ExecutionContext (listings 4.3–4.4)

In [8]:
from scratch_agents.models import Event, ExecutionContext, Message

ctx = ExecutionContext()
e = Event(execution_id=ctx.execution_id, author="user",
          content=[Message(role="user", content="What is 1234 * 5678?")])
ctx.add_event(e)
ctx.increment_step()
print(ctx)

ExecutionContext(execution_id='20dd48c3-3cd3-464f-a6db-310f5abfd260', events=[Event(id='9e9a255e-f136-4080-af55-a2ba4f191c79', execution_id='20dd48c3-3cd3-464f-a6db-310f5abfd260', timestamp=1795505786.264036, author='user', content=[Message(type='message', role='user', content='What is 1234 * 5678?')])], current_step=1, state={}, final_result=None)


### Break it on purpose: Event and ExecutionContext

In [2]:
from scratch_agents.models import Event, ExecutionContext, Message

#### Factory vs Plain default

In [3]:
e1 = Event(execution_id="x", author="user")
e2 = Event(execution_id="x", author="user")
print(e1.id == e2.id)

False


#### Mutable default

In [4]:
from dataclasses import dataclass

@dataclass
class Bad:
    events: list = []

ValueError: mutable default <class 'list'> for field events is not allowed: use default_factory

In [6]:
from dataclasses import fields

@dataclass
class Point:
    x: int           # field, required
    y: int = 0       # field, plain default
    label = "p"      # NOT a field: no type hint

print([f.name for f in fields(Point)])   # ['x', 'y']

['x', 'y']


#### Dataclasses don't validate

In [7]:
ctx = ExecutionContext(current_step="five")
print(ctx.current_step < 10)     # what run()'s loop check does

TypeError: '<' not supported between instances of 'str' and 'int'

In [8]:
ctx.increment_step()             # what step() does at the end

TypeError: can only concatenate str (not "int") to str

#### The discriminator at work
Pass a plain dict as content

### Moving Chapter 3 helpers to `tools/utils.py`

Change made: In `function_to_input_schema`, skip a parameter called `context`. Add this as the first line inside the `for param in ...` loop:

```python
if param.name == "context":
    continue
```

`continue` means "skip the rest of this loop round and go to the next parameter".

Why? A tool that wants the context is written `def check_minibar(context, item: str)`. Without the skip, your helper would put `context` into the schema as a required string, and the LLM would try to fill it in. But `context` comes from the agent, never from the LLM (the room-number rule again). The skip hides it from the LLM.

Edit 2:
 listing 3.17 builds required in a separate list comprehension that loops over all parameters again, so `context` still ends up in required. Add the same condition there too.

 ```python
 required = [
    param.name
    for param in signature.parameters.values()
    if param.default == inspect._empty and param.name != "context"
]
```

In [10]:
from scratch_agents.tools.utils import function_to_input_schema

def check_minibar(context, item: str) -> str: ...
print(function_to_input_schema(check_minibar))

{'type': 'object', 'properties': {'item': {'type': 'string'}}, 'required': ['item']}


In [11]:
from scratch_agents.tools.base import BaseTool
BaseTool()

TypeError: Can't instantiate abstract class BaseTool without an implementation for abstract method 'exectue'

### Chechking The bug in listing 4.6 (after typing it as printed first)

In [13]:
from scratch_agents.tools.base import FunctionTool

def calculator(expression: str) -> float:
    """Calculate mathematical expressions."""
    return eval(expression)

calc_tool = FunctionTool(calculator)

In [14]:
from scratch_agents.models import ExecutionContext

ctx = ExecutionContext()
print(calc_tool.name)
print(calc_tool.description)
print(calc_tool.tool_definition)
print(await calc_tool(ctx, expression="1234 * 5678"))

calculator
Calculate mathematical expressions.
{'type': 'function', 'function': {'name': 'calculator', 'description': 'Calculate mathematical expressions.', 'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string'}}, 'required': ['expression']}}}
7006652


In [2]:
from scratch_agents.models import ExecutionContext
ctx = ExecutionContext()

In [3]:
from scratch_agents.tools import calculator, search_web
print(type(calculator))
print(await calculator(ctx, expression="2 + 3"))

<class 'scratch_agents.tools.base.FunctionTool'>
5
